# Lab 4 - Schema Enforcement and Controlled Evolution

Shows rejected writes, controlled new columns, type widening, and Delta column mapping.


In [0]:
%run ./lab4_00_config


In [0]:
from pyspark.sql import functions as F

spark.sql(f"DROP TABLE IF EXISTS {schema_test_table}")
spark.sql(f"CREATE TABLE {schema_test_table} AS SELECT * FROM {silver_curated_table}")

print("Schema test table:", schema_test_table)
spark.table(schema_test_table).printSchema()


In [0]:
# Schema enforcement: this append should be rejected because an extra column is not allowed without schema evolution.
bad_extra_column_df = spark.table(schema_test_table).limit(1).withColumn("unexpected_source_column", F.lit("not allowed"))

try:
    (
        bad_extra_column_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(schema_test_table)
    )
    print("Unexpected success")
except Exception as exc:
    print("Expected schema enforcement failure:")
    print(str(exc)[:1000])


In [0]:
# Controlled schema evolution: the same compatible addition is accepted with mergeSchema.
controlled_new_column_df = spark.table(schema_test_table).limit(1).withColumn("quality_score", F.lit(100).cast("int"))

(
    controlled_new_column_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(schema_test_table)
)

spark.table(schema_test_table).printSchema()


In [0]:
# Type widening example on the isolated schema table.
try:
    spark.sql(f"ALTER TABLE {schema_test_table} SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')")
    spark.sql(f"ALTER TABLE {schema_test_table} ALTER COLUMN duration_value TYPE BIGINT")
    spark.table(schema_test_table).printSchema()
except Exception as exc:
    print("Type widening was not applied in this workspace/runtime:")
    print(str(exc)[:1000])


In [0]:
# Delta column mapping: use a separate table so the main Silver table remains stable.
spark.sql(f"DROP TABLE IF EXISTS {column_mapping_table}")
spark.sql(f"CREATE TABLE {column_mapping_table} AS SELECT * FROM {silver_curated_table}")

spark.sql(f'''
ALTER TABLE {column_mapping_table} SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
''')

spark.sql(f"ALTER TABLE {column_mapping_table} RENAME COLUMN rating TO age_rating")
spark.sql(f"ALTER TABLE {column_mapping_table} DROP COLUMN director")

spark.table(column_mapping_table).printSchema()


## Data Contract Note

Schema evolution accepts compatible structural changes, but it does not understand semantic intent. A data contract should define allowed columns, data types, required fields, owners, and approved evolution rules. This keeps new columns controlled instead of silently accepting producer mistakes.
